In [ ]:
dac_months_count_query = """
WITH aud AS (
    {aud}
), monthly_activity AS (
    SELECT
        d.contact_id :: int AS contact_id,
        d.date_month,
        CASE
            WHEN d.has_transaction_activity = 1
             AND (
                  d.has_mobapp_activity = 1
                  OR d.has_pwa_activity = 1
                  OR d.vcoff_trn_cnt > 0
             )
            THEN 1 ELSE 0
        END :: int AS is_dac_month
    FROM dm.prvdr_dac_monthly d
    JOIN aud USING(contact_id)
    WHERE d.date_month < date_trunc('month', date'{date}')
), window_agg AS (
    SELECT
        contact_id,
        SUM(is_dac_month) :: int AS dac_months_count,
        SUM(
            CASE
                WHEN is_dac_month = 1
                 AND date_month >= date_trunc('month', date'{date}') - INTERVAL '3 month'
                THEN 1 ELSE 0
            END
        ) :: int AS dac_months_last_3,
        SUM(
            CASE
                WHEN is_dac_month = 1
                 AND date_month >= date_trunc('month', date'{date}') - INTERVAL '6 month'
                THEN 1 ELSE 0
            END
        ) :: int AS dac_months_last_6,
        SUM(
            CASE
                WHEN is_dac_month = 1
                 AND date_month >= date_trunc('month', date'{date}') - INTERVAL '12 month'
                THEN 1 ELSE 0
            END
        ) :: int AS dac_months_last_12
    FROM monthly_activity
    GROUP BY contact_id
)
SELECT
    aud.contact_id :: int,
    COALESCE(w.dac_months_count, 0) :: int AS dac_months_count,
    COALESCE(w.dac_months_last_3, 0) :: int AS dac_months_last_3,
    COALESCE(w.dac_months_last_6, 0) :: int AS dac_months_last_6,
    COALESCE(w.dac_months_last_12, 0) :: int AS dac_months_last_12
FROM aud
LEFT JOIN window_agg w USING(contact_id)
"""

In [ ]:
df["dac_share_last_12"] = df["dac_months_last_12"] / 12

df["is_stable_dac"] = (df["dac_months_last_12"] >= 10).astype(int)
df["is_regular_dac"] = df["dac_months_last_12"].between(6, 9).astype(int)
df["is_unstable_dac"] = df["dac_months_last_12"].between(2, 5).astype(int)
df["is_new_dac"] = (df["dac_months_last_12"] == 1).astype(int)